# E09 — o vigia vê dependência?

A primeira pergunta da travessia (F1, do contrato): **latência e falso alarme contra uma mudança que
existe só na dependência, sem mudança de margem.** O refutador dela é explícito — a mudança sendo
detectada pelo vigia de margem, sem instrumento extra — e este caderno o testa.

**A construção.** Duas pernas com a mesma lei e o mesmo tamanho, e no meio só o **par** muda: a
correlação entre elas passa de um valor a outro. Cada margem é idêntica do começo ao fim, e nenhum
instrumento que leia uma perna por vez tem o que ver.

**O que se mede.** A taxa de rompimento de cada perna antes e depois (a cegueira), a contagem
conjunta antes e depois, e a troca do instrumento que vê: orçamento de falso alarme contra
latência, com quarenta sementes por limiar.

**A pergunta é sobre estrutura, e não sobre o mundo** (AGENTS.md §8.5): a mudança é posta de
propósito numa cópula gaussiana, e é isso que permite datar o instante e medir a latência.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E09_dependencia_vigiada.json.

In [1]:
# <- brinque com: DIAS, JANELA, POSTO, BLOCO, RHO_ANTES, RHO_DEPOIS, QUANDO, LIMIARES, SEMENTES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dependencia, graficos, mudanca, promessa, vigia

DIAS = 6000
JANELA, POSTO, BLOCO = 252, 13, 60
CAUDA = POSTO / JANELA
RHO_ANTES, RHO_DEPOIS, QUANDO = 0.2, 0.8, 3000
LIMIARES = (2, 3, 4, 5, 6)
SEMENTES = 40

indice = pd.RangeIndex(DIAS)
print("frevolab %s | %d dias | o par muda no dia %d (de %.1f para %.1f)" % (
    frevolab.VERSAO, DIAS, QUANDO, RHO_ANTES, RHO_DEPOIS))

frevolab 0.1.0 | 6000 dias | o par muda no dia 3000 (de 0.2 para 0.8)


## A margem não vê

In [2]:
# A margem: cada perna rompe na mesma taxa antes e depois da mudanca do par.
a, b = mudanca.dependencia(DIAS, np.random.default_rng(11), rho_antes=RHO_ANTES,
                           rho_depois=RHO_DEPOIS, quando=QUANDO)
perna_a, perna_b = pd.Series(a, index=indice), pd.Series(b, index=indice)
rompe_a = promessa.violacoes(perna_a, JANELA, CAUDA)
rompe_b = promessa.violacoes(perna_b, JANELA, CAUDA)
corte_tempo = QUANDO - JANELA
print("perna A: %.4f antes | %.4f depois" % (rompe_a.iloc[:corte_tempo].mean(),
                                             rompe_a.iloc[corte_tempo:].mean()))
print("perna B: %.4f antes | %.4f depois" % (rompe_b.iloc[:corte_tempo].mean(),
                                             rompe_b.iloc[corte_tempo:].mean()))
print("diferenca entre antes e depois, perna A: %+.5f" % (
    rompe_a.iloc[corte_tempo:].mean() - rompe_a.iloc[:corte_tempo].mean()))
conjunto = dependencia.bloco_conjunto(perna_a, perna_b, JANELA, CAUDA, BLOCO)
print("contagem conjunta: %.2f antes | %.2f depois" % (conjunto.iloc[:corte_tempo].mean(),
                                                       conjunto.iloc[corte_tempo:].mean()))

perna A: 0.0528 antes | 0.0523 depois
perna B: 0.0520 antes | 0.0513 depois
diferenca entre antes e depois, perna A: -0.00043
contagem conjunta: 0.41 antes | 1.56 depois


## O instrumento conjunto vê, e paga

In [3]:
# A troca do instrumento que ve: orcamento de falso alarme contra latencia, com replicacao.
linhas = []
for limiar in LIMIARES:
    episodios, latencias, sem_ver = [], [], 0
    for semente in range(300, 300 + SEMENTES):
        falso = mudanca.dependencia(DIAS, np.random.default_rng(semente), rho_antes=RHO_ANTES,
                                    rho_depois=RHO_ANTES, quando=QUANDO)
        c0 = dependencia.bloco_conjunto(pd.Series(falso[0], index=indice),
                                        pd.Series(falso[1], index=indice), JANELA, CAUDA, BLOCO)
        episodios.append(len(promessa.episodios_acima(c0, limiar)))
        verdadeiro = mudanca.dependencia(DIAS, np.random.default_rng(semente), rho_antes=RHO_ANTES,
                                         rho_depois=RHO_DEPOIS, quando=QUANDO)
        c1 = dependencia.bloco_conjunto(pd.Series(verdadeiro[0], index=indice),
                                        pd.Series(verdadeiro[1], index=indice), JANELA, CAUDA, BLOCO)
        alarmes = pd.Series((c1 > limiar).to_numpy(), index=c1.index)
        atraso = vigia.latencia(alarmes, corte_tempo)
        if np.isnan(atraso):
            sem_ver += 1
        else:
            latencias.append(atraso)
    anos = len(conjunto) / 252.0
    media = float(np.mean(episodios))
    linhas.append({"limiar": limiar, "anos por alarme": (anos / media) if media > 0 else float("inf"),
                   "latencia (dias)": float(np.median(latencias)) if latencias else float("nan"),
                   "nao viu": sem_ver})
tabela = pd.DataFrame(linhas).set_index("limiar")
print(tabela.round(1).to_string())

        anos por alarme  latencia (dias)  nao viu
limiar                                           
2                  17.0            354.5        0
3                 150.5            524.0        0
4                   inf            907.0        3
5                   inf           1726.5       20
6                   inf           1412.5       34


## O controle: o mundo em que o par não muda

In [4]:
# O controle: no mundo em que a dependencia NAO muda, o instrumento nao pode achar mudanca.
achados = []
for semente in range(300, 300 + SEMENTES):
    parado = mudanca.dependencia(DIAS, np.random.default_rng(semente), rho_antes=RHO_ANTES,
                                 rho_depois=RHO_ANTES, quando=QUANDO)
    c = dependencia.bloco_conjunto(pd.Series(parado[0], index=indice),
                                   pd.Series(parado[1], index=indice), JANELA, CAUDA, BLOCO)
    alarmes = pd.Series((c > int(tabela.index[1])).to_numpy(), index=c.index)
    atraso = vigia.latencia(alarmes, corte_tempo)
    achados.append(0 if np.isnan(atraso) else 1)
print("no mundo em que o par NAO muda, com o limiar %d: alarmou em %d de %d mundos (%.0f%%)"
      % (int(tabela.index[1]), sum(achados), SEMENTES, 100 * np.mean(achados)))

no mundo em que o par NAO muda, com o limiar 3: alarmou em 4 de 40 mundos (10%)


## A figura

In [5]:
# Figura 1: a contagem conjunta, com o limiar e o dia da mudanca.
fig, eixo = plt.subplots(figsize=(9.4, 4.2))
eixo.plot(conjunto.index, conjunto.to_numpy(), color="#1f4e79", lw=0.9)
# A linha marca onde a janela da margem termina (QUANDO - JANELA), que e o ponto de corte
# da comparacao antes/depois --- e nao onde o par muda, que e QUANDO, 252 dias adiante.
eixo.axvline(corte_tempo, color="#b03a2e", ls="--", lw=1.4, label="a janela da margem termina aqui")
eixo.axhline(3, color="#7f7f7f", ls=":", lw=1.4, label="o limiar do instrumento conjunto")
eixo.set_xlabel("dias")
eixo.set_ylabel("dias com as duas rompidas, nos %d anteriores" % BLOCO)
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E09_dependencia_vigiada", 1)
plt.close(fig)
print("contagem conjunta: pico antes %.0f | pico depois %.0f" % (
    conjunto.iloc[:corte_tempo].max(), conjunto.iloc[corte_tempo:].max()))

contagem conjunta: pico antes 2 | pico depois 7


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Uma linha quase colada no chão por dois mil e setecentos dias, com picos isolados de
um ou dois, e a linha vertical da mudança. Depois dela a linha fica visivelmente mais densa: os
picos passam a subir até o dobro da altura e ficam mais próximos uns dos outros, ainda com vales
entre eles. A linha pontilhada do limiar, no três, atravessa o painel inteiro: antes da mudança ela
é tocada uma vez, depois é cruzada várias. O que o eixo engana: o painel tem escala curta (zero a
sete) e a linha é serrilhada, de modo que a mudança aparece como "mais dentes" e não como salto — e
quem procura um degrau na figura não o encontra, porque o que mudou foi a frequência, não o nível.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
primeiro = tabela.iloc[0]
segundo = tabela.iloc[1]
resultado = {
    "vizinhanca_dias": DIAS,
    "vizinhanca_rho_antes": RHO_ANTES,
    "vizinhanca_rho_depois": RHO_DEPOIS,
    "vizinhanca_quando": QUANDO,
    "vizinhanca_sementes": SEMENTES,
    "vizinhanca_taxa_a_antes": float(rompe_a.iloc[:corte_tempo].mean()),
    "vizinhanca_taxa_a_depois": float(rompe_a.iloc[corte_tempo:].mean()),
    "vizinhanca_taxa_b_antes": float(rompe_b.iloc[:corte_tempo].mean()),
    "vizinhanca_taxa_b_depois": float(rompe_b.iloc[corte_tempo:].mean()),
    "vizinhanca_conjunto_antes": float(conjunto.iloc[:corte_tempo].mean()),
    "vizinhanca_conjunto_depois": float(conjunto.iloc[corte_tempo:].mean()),
    "vizinhanca_limiar_dois_anos": float(primeiro["anos por alarme"]),
    "vizinhanca_limiar_dois_latencia": float(primeiro["latencia (dias)"]),
    "vizinhanca_limiar_tres_anos": float(segundo["anos por alarme"]),
    "vizinhanca_limiar_tres_latencia": float(segundo["latencia (dias)"]),
    "vizinhanca_limiar_quatro_latencia": float(tabela.iloc[2]["latencia (dias)"]),
    "vizinhanca_limiar_quatro_nao_viu": int(tabela.iloc[2]["nao viu"]),
    "vizinhanca_limiar_cinco_nao_viu": int(tabela.iloc[3]["nao viu"]),
    "vizinhanca_limiar_seis_nao_viu": int(tabela.iloc[4]["nao viu"]),
    "vizinhanca_controle_alarmou_pct": float(100 * np.mean(achados)),
}
caminho = Path("lab/resultados/E09_dependencia_vigiada.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E09_dependencia_vigiada.json gravado | 20 grandezas
